# Narrative Similarity Retrieval Pipeline

**Thesis**: Quantifying the Impact of Temporal and Causal Structures on Narrative Similarity Retrieval

This notebook builds the event-based narrative representations for one dataset arm and runs every step up to the lexical baselines. The retrieval evaluation lives in a separate notebook, `Evaluation.ipynb`.

**Run this notebook twice, once per arm:**
1. **Non-pseudonymized arm:** run Section 2a, which prints the `export EXPERIMENT_DIR` / `SPLIT` lines for the non-pseudonymized folder; paste those into your terminal, then work through the rest of the notebook.
2. **Pseudonymized arm:** run Section 2b, 2b prints the `export` lines for the pseudonymized folder; paste those into your terminal, then work through the rest of the notebook again.

Each pass produces one `experiment.jsonl` within their own folder. `Evaluation.ipynb` then reads both folders and computes the matched, side-by-side results. 

**Pipeline steps:**
1. **Event detection (MAVEN):** train the BERT+CRF event extractor (Section 1).
2. **Dataset subsetting:** build the eligible Tell Me Again! corpus; Section 2a for the non-pseudonymized arm, Section 2b for the pseudonymized arm.
3. **Event extraction:** run BERT+CRF over the subset (Section 3), then drop summaries with fewer than 5 events and re-enforce the at-least-2-summaries-per-work floor (Section 3.5).
4. **Relation annotation:** Llama-3-8B zero-shot temporal, causal, and joint temporo-causal relations (Section 4), then complete-case and hallucinated-relation filtering (Sections 4.5 and 4.6).
5. **Linearization:** convert events and relations to text per condition (Section 5), then drop linearized-overflow summaries (Section 5.5).
6. **Embedding:** encode every condition with E5-Mistral, Qwen3-0.6B, and StoryEmbed (Section 6).

Each step writes its output under a per-experiment folder `data/experiments/<experiment_name>/`, so runs with different arms / prompts / embedders / sample sizes never overwrite each other. Shared inputs (the BERT+CRF checkpoint, the parsed TMA cache, raw datasets) stay global under `data/`.

## 0. Setup & Imports

### Environments

This project uses **three** Python environments. Two of them live in this repo and look almost identical: `venv` (local) and `.venv` (on the cluster). The leading dot keeps the local `venv` and the cluster `.venv` from being mixed up. 
| Environment | Path | Python | Used for |
|---|---|---|---|
| **Pipeline** | `venv/` (your machine) | 3.14 | This notebook and every local step. Use this unless a step says otherwise. |
| **BERT+CRF** | `models/bert_crf/.venv-maven-train` (your machine) | 3.9 | Only training (Section 1.2) and event extraction (Section 3). Pinned to transformers 4.18. Created in Section 1.2. |
| **Cluster** | `.venv/` (on Snellius) | 3.x | Only the *cluster option* of Llama (Section 4) and embedding (Section 6). Created on Snellius (see Section 4). |

**Here is a summary of all the steps, in the order to run them, each with the environment shown (each section also has its own instructions):**

| # | Step | Environment | Run from |
|---|---|---|---|
| 1 | Set up the pipeline env, select it as the kernel (commands below) | Pipeline | n/a |
| 2 | Load MAVEN (Section 1.1) | Pipeline | notebook |
| 3 | Train BERT+CRF (Section 1.2) | BERT+CRF | terminal |
| 4 | Subset Tell Me Again! (Section 2) | Pipeline | notebook |
| 5 | Extract events (Section 3) | BERT+CRF | terminal |
| 6 | Pre-annotation processing (Section 3.5) | Pipeline | notebook |
| 7 | Annotate relations with Llama (Section 4) | Pipeline *(local)* or Cluster *(Snellius)* | terminal |
| 8 | Pre-embedding processing + linearize (Section 4.5–5) | Pipeline | notebook |
| 9 | Embed (Section 6) | Pipeline *(local)* or Cluster *(Snellius)* | terminal |
| 10 | Lexical baselines on the matched set, both arms | Pipeline | notebook (`Evaluation.ipynb`) |
| 11 | Evaluation of both arms, run once after both passes | Pipeline | notebook (`Evaluation.ipynb`) |

**Set up the pipeline environment now** (from the project root), then select `venv` as this notebook's kernel:
```bash
python3 -m venv venv                 # this experiment used Python 3.14.5 in local, 3.13.5 in cluster
source venv/bin/activate             # Windows: venv\Scripts\activate
pip install --upgrade pip
pip install -r requirements.txt      # the repo-root file, NOT models/bert_crf/requirements.txt
```

**Also request Llama-3-8B access**  We use the *gated* model `meta-llama/Meta-Llama-3-8B-Instruct`, and it's tokenizer throughout the notebook. Continue this pipeline after approval, it might take few days. 
```bash
hf auth login    # paste a read-scope token from https://huggingface.co/settings/tokens. Do it for both local and cluster login node. 


In [ ]:
# Import the necessary libraries and modules
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import json
import pickle
from tell_me_again import StoryDataset
from transformers import AutoTokenizer
import collections
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import random
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from scipy import sparse
from collections import defaultdict
import yaml
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re

# Define the project root
ROOT = Path(os.getcwd()).parent

# Make sure the project root is in the Python path so we can import from `src`
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))  

# Import the function to generate new experiment names
from src.experiment_paths import new_experiment_name

# Define data paths
DATA_RAW = ROOT / "data" / "raw"
DATA_INTERMEDIATE = ROOT / "data" / "intermediate"
#DATA_RESULTS = ROOT / "data" / "results"
DATA_EXPERIMENTS = ROOT / "data" / "experiments"

# Create intermediate/results/experiments dirs if they don't exist
DATA_INTERMEDIATE.mkdir(parents=True, exist_ok=True)
#DATA_RESULTS.mkdir(parents=True, exist_ok=True)
DATA_EXPERIMENTS.mkdir(parents=True, exist_ok=True)

# Print the paths to verify
print(f"Project root: {ROOT}")
print(f"Raw data:     {DATA_RAW}")
print(f"Intermediate: {DATA_INTERMEDIATE}")
#print(f"Results:      {DATA_RESULTS}")
print(f"Experiments:  {DATA_EXPERIMENTS}")

## 1. Event Detection (MAVEN)

### 1.0 Download the MAVEN Dataset

MAVEN is not redistributed in this repo — download it once and place the splits
under `data/raw/MAVEN/`.

1. Open the MAVEN dataset repository: https://github.com/THU-KEG/MAVEN-dataset
2. Follow the download link in its README (the data ships as a zip) and unzip it.
3. Place the three split files under **data/raw/MAVEN** folder. 




### 1.1 Load MAVEN Dataset

MAVEN contains 4,480 Wikipedia documents annotated with 168 event types. Each document has:
- `content`: list of sentences with tokenized words
- `events`: list of event types, each with one or more trigger mentions (word + sentence + offset)
- `negative_triggers`: words that look like events but aren't

Splits: 2,913 train / 710 valid / 857 test documents.

In [ ]:
# Define a function to load JSONL files
def load_jsonl(path):
    with open(path, "r") as f:
        return [json.loads(line) for line in f]

# Load MAVEN data
MAVEN_DIR = DATA_RAW / "MAVEN"
train_maven = load_jsonl(MAVEN_DIR / "train.jsonl")
valid_maven = load_jsonl(MAVEN_DIR / "valid.jsonl")
test_maven = load_jsonl(MAVEN_DIR / "test.jsonl")

# Print dataset sizes
print(f"Train: {len(train_maven)} docs")
print(f"Valid: {len(valid_maven)} docs")
print(f"Test:  {len(test_maven)} docs")

**Goal:** Print the first 3 events of the first MAVEN document to review the event structure (trigger word, event type, character offsets) the model is trained to detect.

In [ ]:
# Quick look at a single document to understand the structure (tweak the "0" to see random documents)
doc = train_maven[2]

#Print the meta data
print(f"Title: {doc['title']}")
print(f"Sentences: {len(doc['content'])}")
print(f"Event types: {len(doc['events'])}")
print(f"Negative triggers: {len(doc['negative_triggers'])}")

# Print the full story text in sentences for readability
print("\nStory text:")
for sent_id, s in enumerate(doc["content"]):
    print(f"  [{sent_id}] {s['sentence']}")

# Print details of the first 3 events of the first document to understand the event structure
for i, evt in enumerate(doc["events"][:3], start=1):
    print(f"\nEvent {i}:")
    print(f"  Type: {evt['type']} (id: {evt['type_id']})")
    for mention in evt["mention"]:
        print(f"  Trigger: '{mention['trigger_word']}' in sentence {mention['sent_id']}, offset {mention['offset']}")

### 1.2 Train BERT+CRF

The training code (`models/bert_crf/`) is based on the [MAVEN baseline](https://github.com/THU-KEG/MAVEN-dataset/tree/main/baselines/BERT%2BCRF). Refer to their page for details. Their original code targeted `transformers==2.6.0` / `torch==1.2.0`, but those versions have no arm64 (Apple Silicon) wheels, and Apple Macbook Air M2 was used during for this thesis. We use `transformers==4.18.0` and `tokenizers==0.11.6` , which are the oldest compatible combination with arm64 wheels, and we patched few lines (see `PATCHED` comments in `bert_crf.py` and `run_maven.py`). No training logic was changed.

**Notes**:
- The venv is created **inside `models/bert_crf/`** (not at project root) so it exists next to the code that imports it. 
- Python must be **3.9**. `transformers==4.18.0` has no wheels for Python 3.11+, so creating the venv with a newer interpreter will use a newer transformers release, and the custom `BertCRFForTokenClassification` subclass will break against transformers 5.x internals (`all_tied_weights_keys` attribute error). We invoke `/usr/bin/python3` explicitly because Apple's CommandLineTools was still shipping [Python 3.9.6 ](https://www.python.org/downloads/release/python-396/)  at the time of experimentation. You can download this version and use it's "python3" path, instead of `/usr/bin/python3`. 

**Setup (from project root, in terminal):**
```bash
# Deactivate the previously activated venv, if it's activated in the terminal already
deactivate

# Change directory to the bert+crf model's directory
cd models/bert_crf

# Create the venv based on Python 3.9 (in our case, it was on usr/bin/python3)
/usr/bin/python3 -m venv .venv-maven-train

# Activate the custom virtual environment
source .venv-maven-train/bin/activate

# Verify before installing — abort if this is not 3.9.x
python -V

# Upgrade the pip if needed
pip install --upgrade pip

# Install the requirements.txt under model/bert_crf directory
pip install -r requirements.txt  

# Train BERT+CRF model on MAVEN dataset (run the code within `models/bert_crf/`):
python run_maven.py \
    --data_dir ../../data/raw/MAVEN/ \
    --model_type bertcrf \
    --model_name_or_path bert-base-uncased \
    --output_dir ../../data/intermediate/models/bert_crf/ \
    --max_seq_length 128 \
    --do_lower_case \
    --per_gpu_train_batch_size 16 \
    --per_gpu_eval_batch_size 16 \
    --gradient_accumulation_steps 8 \
    --learning_rate 5e-5 \
    --num_train_epochs 5 \
    --save_steps 100 \
    --logging_steps 100 \
    --seed 0 \
    --do_train \
    --do_eval \
    --evaluate_during_training \
    --overwrite_output_dir

# Optional: Final evaluation against "valid.jsonl" 
# Although it runs automatically at the end of the command above; can also be re-run standalone using the saved checkpoint without re-training the model.
# Results are written within `data/intermediate/models/bert_crf/eval_results.txt` with `f1`, `precision`, `recall`, `loss` on the MAVEN validation split.
python run_maven.py \
    --data_dir ../../data/raw/MAVEN/ \
    --model_type bertcrf \
    --model_name_or_path bert-base-uncased \
    --output_dir ../../data/intermediate/models/bert_crf/ \
    --max_seq_length 128 \
    --do_lower_case \
    --per_gpu_eval_batch_size 16 \
    --seed 0 \
    --do_eval
    
# When done, deactivate the environment. 
# Rest of the code uses the first environment we created at the beginning of this pipeline (venv) or cluster environment, namely .venv, if local compute is not possible.
deactivate
```

After training completes, the checkpoint directory should contain:
- `pytorch_model.bin` — model weights
- `config.json` — BERT config with num_labels=337 (168 event types × 2 for B/I tags + O)
- `vocab.txt` — BERT tokenizer vocabulary
- `eval_results.txt` — final metrics on `valid.jsonl`

**Goal:** Verify the checkpoint directory contains the necessary files (note that eval_results.txt is just a metrics report, not a model artifact that pipeline needs.)

In [ ]:
# Define the path to the BERT-CRF checkpoint
BERT_CRF_CHECKPOINT = DATA_INTERMEDIATE / "models" / "bert_crf" 

# Verify checkpoint exists
required_files = ["pytorch_model.bin", "config.json", "vocab.txt"]
isCheckPointMissing = [f for f in required_files if not (BERT_CRF_CHECKPOINT / f).exists()]

# Print checkpoint status and file sizes (config.json can be listed with 0.0 MB, it should be still there in the folder)
if isCheckPointMissing:
    print(f"Checkpoint not found at: {BERT_CRF_CHECKPOINT}")
    print(f"Missing files: {isCheckPointMissing}")
    print("Please run the training commands from Section 1.2 first.")
else:
    print(f"Checkpoint found at: {BERT_CRF_CHECKPOINT}")
    for f in required_files:
        size_mb = (BERT_CRF_CHECKPOINT / f).stat().st_size / 1e6
        print(f"  {f}: {size_mb:.1f} MB")

## 2. Dataset Subsetting

**Prerequisite (one-time):** Download the Tell Me Again! dataset zip from https://github.com/uhh-lt/tell-me-again and place it at exactly:

```
data/raw/TellMeAgain/tell_me_again_v1.zip
```

The cell below reads the zip from this path (see the **Setup** section in the `README`).

**Goal:** Build the eligible [Tell Me Again!](https://github.com/uhh-lt/tell-me-again/tree/main/tell_me_again) retrieval corpus by applying filters (see Thesis) to the official split selected by `SPLIT` (the constant defined in the next cell). The split **test** is being used in this experiment (see Thesis)

The output is a flat JSONL where each row is one summary, keyed by `wikidata_id` (the relevance-cluster key for retrieval evaluation).

Down-Sampling the Full TMA Dataset:
1. Restrict to the chosen `SPLIT` (i.e., test). 
2. Drop stories with no Wikidata genre.
3. Drop near-duplicate translated summaries (cosine > `DEDUP_COSINE_THRESHOLD` against EN original) and summaries shorter than `MIN_SENTENCES`.
4. Drop summaries above `MAX_TOKENS` (With respect to Llama-3-8B context window).
5. Drop summaries containing any sentence over `BERT_MAX_SUBWORDS_PER_SENTENCE` BERT subwords (BERT+CRF inference limit, see their source code).
6. Drop stories left with fewer than `MIN_DEDUPED_EN_SUMMARIES` surviving English summaries (so that each story has at least 2 works to be eligible for narrative similarity retrieval task)
7. Optionally limit the `N_STORIES` to an arbitrary number for local development. Keep it **None** for full data set. 

Output: `$EXPERIMENT_DIR/tma_subset.{SPLIT}.jsonl`.

In [ ]:
# Dataset Subsetting parameters
N_STORIES: int | None = None         # max number of stories to process (None = all)
MIN_SENTENCES: int = 0               # If a story has <MIN_SENTENCES sentences, it is dropped. It's set to 0, since main filtering was done via number of events later on
MAX_TOKENS: int = 8192               # Llama-3-8B context window
MIN_DEDUPED_EN_SUMMARIES: int = 2    # min surviving EN summaries per story (relevance clusters need >=2 summaries to be retained per story, otherwise the story is dropped)
DEDUP_COSINE_THRESHOLD: float = 0.6  # translation-dedup threshold (matches with the prior TMA work)
MIN_EVENTS_FOR_RELATIONS  = 5        # a story needs >=5 events to yield enough relations for evaluation, based on previous research (see Thesis) 

# Sentence-level BERT budget: sentences with >126 subwords are dropped (~ 0.75% of sentences, per EDA), since BERT was trained with 126 token limit per sentence due to computational limitations. 
BERT_MAX_SUBWORDS_PER_SENTENCE: int = 126
BERT_TOKENIZER_ID: str = "bert-base-uncased"

# Tokenizer: load once and reuse for all summaries (Llama-3-8B-Instruct tokenizer)
LLAMA_TOKENIZER_ID: str = "meta-llama/Meta-Llama-3-8B-Instruct"

# Select the TMA split to process (Options: train/valid/test)
SPLIT: str = "test"

# Print the subsetting parameters for verification
print(f"N_STORIES                       =               {N_STORIES}")
print(f"MIN_SENTENCES                   =               {MIN_SENTENCES}")
print(f"MAX_TOKENS                      =               {MAX_TOKENS}")
print(f"MIN_DEDUPED_EN_SUMMARIES        =               {MIN_DEDUPED_EN_SUMMARIES}")
print(f"DEDUP_COSINE_THRESHOLD          =               {DEDUP_COSINE_THRESHOLD}")
print(f"BERT_MAX_SUBWORDS_PER_SENTENCE  =               {BERT_MAX_SUBWORDS_PER_SENTENCE}")
print(f"SPLIT                           =               {SPLIT}")

### 2a. Non-Pseudonymized subset (Do not run for pseudonymzed arm)
**Goal:** Apply the filters defined above to the selected `SPLIT` and write the eligible retrieval corpus to `tma_subset.{SPLIT}.jsonl` inside a new per-experiment folder.

**Note:** The code below mints a fresh experiment folder every time it runs, named `experiment_<split>_<n_summaries>_<date+time>` (split, number of summaries kept, current date+time). This lets multiple experiments coexist side by side and keeps each run easy to track and reproduce.

**Then:** When it finishes, the cell prints two `export` lines (`EXPERIMENT_DIR` and `SPLIT`). Paste them into your local/cluster terminal before the terminal-run steps (Sections 3, 4, 6), and re-run them if you lose the session or re-run the pipeline with different parameters.

**Important:** This builds only the non-pseudonymized arm. The Evaluation notebook compares the non-pseudonymized and pseudonymized arms together, using only the summaries that survive in both. To reach those final results you must build both arms (this cell and the pseudonymized subset in 2b) and run the whole pipeline for each, then run the Evaluation notebook

In [ ]:
# Define the path to the raw TellMeAgain zip and the intermediate cache for parsed StoryDataset
TMA_ZIP_PATH = DATA_RAW / "TellMeAgain" / "tell_me_again_v1.zip"
TMA_PARSED_CACHE = DATA_INTERMEDIATE / "_cache_storydataset.pkl"

# Tokenizer: load once and reuse for all summaries (Llama-3-8B-Instruct tokenizer)
if "_tokenizer" not in globals():
    print("Loading Llama-3 tokenizer...")
    _tokenizer = AutoTokenizer.from_pretrained(LLAMA_TOKENIZER_ID)

# Helper function to count Llama tokens in a text
def n_llama_tokens(text: str) -> int:
    return len(_tokenizer.encode(text, add_special_tokens=False))

# BERT tokenizer for the sentence-length filter (matches BERT+CRF training: do_lower_case=True)
if "_bert_tokenizer" not in globals():
    print("Loading BERT tokenizer...")
    _bert_tokenizer = AutoTokenizer.from_pretrained(BERT_TOKENIZER_ID, do_lower_case=True)

# Define a helper function to compute the max BERT subword count across sentences in a summary, which is used for the sentence-length filter.
def max_bert_subwords(sentences):
    """Max subword count across the sentences of a summary (no specials)."""
    return max(
        (len(_bert_tokenizer.encode(s, add_special_tokens=False)) for s in sentences),
        default=0,
    )

# Since TMA StoryDataset is downloaded az a zip and parsing is expensive, include them into in-memory cache for reuse. 
if "_ds" not in globals():
    if TMA_PARSED_CACHE.exists():
        print(f"Loading StoryDataset from disk cache ({TMA_PARSED_CACHE.name})...")
        with open(TMA_PARSED_CACHE, "rb") as f:
            _ds = pickle.load(f)
    else:
        print("Parsing TellMeAgain zip...")
        _ds = StoryDataset(data_path=TMA_ZIP_PATH)
        TMA_PARSED_CACHE.parent.mkdir(parents=True, exist_ok=True)
        with open(TMA_PARSED_CACHE, "wb") as f:
            pickle.dump(_ds, f)
        print(f"  -> cached to {TMA_PARSED_CACHE} "
              f"({TMA_PARSED_CACHE.stat().st_size / 1e6:.1f} MB)")
        

# Since Official TMA package has a reported (yet not addressed) bug, we unpack the zip here
# Patch tell_me_again==0.1.0: its perform_splits has a known bug (`# TODO: fix path`).
# __init__ reads stories from *inside* the zip, but perform_splits does
# open(self.data_path / "<split>_stories.csv"), which treats the zip file as a
# directory -> NotADirectoryError. The split id lists live inside the zip, so read them there.
import zipfile
from tell_me_again import StoryDataset

def _perform_splits_from_zip(self):
    zf = zipfile.ZipFile(self.data_path)
    split_ids = {}
    for split in ["train", "dev", "test"]:
        with zf.open(f"{split}_stories.csv") as fh:
            split_ids[split] = [l.decode("utf-8").strip() for l in fh if l.strip()]
    return {
        split: self.__class__(
            data_path=self.data_path,                      # keep the zip path so __init__
            stories={i: self.stories[i]                    # early-returns (no re-parse,
                     for i in split_ids[split]             #  no 1.6 GB re-download)
                     if i in self.stories},
        )
        for split in ["train", "dev", "test"]
    }

StoryDataset.perform_splits = _perform_splits_from_zip

# Restrict the dataset to the chosen SPLIT (which is test for this thesis)
_stories = _ds.perform_splits()[SPLIT]
print(f"  [1] {SPLIT} split:{' ' * (28 - len(SPLIT))}{len(_stories):>6} stories")

# Drop stories with no Wikidata genre
stories_with_genre = {wid: s for wid, s in _stories.stories.items() if s.genres}
print(f"  [2] with >=1 Wikidata genre:           {len(stories_with_genre):>6} stories")

# Per-story: dedup translations + min_sentences (handled by package), then per-summary token filter, then min surviving EN summaries.
n_stories_kept = 0
n_summaries_kept = 0
n_summaries_dropped_tokens = 0
n_summaries_dropped_long_sentence = 0
rows = []  # buffer: written to JSONL at the end

# Deterministic ordering for the N_STORIES cap so that same stories are kept across different runs with the same parameters, and to make the subsetting process reproducible and debuggable.
sorted_wids = sorted(stories_with_genre.keys())

# For each story (until N_STORIES cap is reached if set),
# - get the EN summaries that survive deduplication and min_sentences, 
# - then apply the token and sentence-length filters, 
# - and keep the story if it has >=MIN_DEDUPED_EN_SUMMARIES surviving summaries.
for wid in sorted_wids:
    story = stories_with_genre[wid]
    ids, summaries = story.get_all_summaries_en(
        max_similarity=DEDUP_COSINE_THRESHOLD,
        min_sentences=MIN_SENTENCES,
    )
    surviving = []
    for summary_id, text in zip(ids, summaries):
        n_tok = n_llama_tokens(text)
        if n_tok > MAX_TOKENS:
            n_summaries_dropped_tokens += 1
            continue
        sentences = story.sentences[summary_id]
        if max_bert_subwords(sentences) > BERT_MAX_SUBWORDS_PER_SENTENCE:
            n_summaries_dropped_long_sentence += 1
            continue
        surviving.append({
            "wikidata_id": wid,
            "summary_id": summary_id,
            "lang": summary_id,               
            "text": text,
            "sentences": sentences,
            "n_sentences": len(sentences),
            "n_tokens": n_tok,
            "genres": story.genres,
            "split": SPLIT,
        })
        
    # If the story doesn't have enough surviving EN summaries after all filters, drop the story entirely
    if len(surviving) < MIN_DEDUPED_EN_SUMMARIES:
        continue
    
    # If we reach this point, the story has enough surviving summaries to be kept, so we add them to the buffer and update the counters.
    n_stories_kept += 1
    n_summaries_kept += len(surviving)
    rows.extend(surviving)

    # If N_STORIES is reached, stop processing more stories
    if N_STORIES is not None and n_stories_kept >= N_STORIES:
        break

# Print intermediate stats after all filtering steps but before writing the JSONL, so we can see the impact of the filters without needing to re-run the entire cell with different parameters.
print(f"  [3-5] after dedup + length + cluster:  {n_stories_kept:>6} stories, {n_summaries_kept} summaries")
print(f"        ({n_summaries_dropped_tokens} summaries dropped for >{MAX_TOKENS} Llama tokens)")
print(f"        ({n_summaries_dropped_long_sentence} summaries dropped for sentence >{BERT_MAX_SUBWORDS_PER_SENTENCE} BERT subwords)")

# Create a new experiment folder for this specific subsetting configuration, using the helper function to generate a unique name based on the SPLIT and number of summaries.
EXPERIMENT_NAME = new_experiment_name(SPLIT, len(rows))
EXPERIMENT_DIR  = DATA_EXPERIMENTS / EXPERIMENT_NAME
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
(EXPERIMENT_DIR / "logs").mkdir(parents=True, exist_ok=True)
TMA_SUBSET_PATH = EXPERIMENT_DIR / f"tma_subset.{SPLIT}.jsonl"   # filename unchanged

# Save the subsetted dataset to a new JSONL file in the experiment folder.
TMA_SUBSET_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(TMA_SUBSET_PATH, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Print final summary of the subsetting results.
print(f"- Wrote {len(rows)} summaries across {n_stories_kept} stories")
print(f"- Experiment Name:                         {EXPERIMENT_NAME}")
print(f"- Location of the subsetted TMA dataset:   {TMA_SUBSET_PATH}")
print()

# Display the next steps instructions as Markdown for better visibility in the notebook, since these are important to run before proceeding with the Local/Cluster steps.
display(Markdown(f"**IMPORTANT:** For the Local/Cluster steps below, run this first within Local/Cluster terminal: (And re-run them if you loose the terminal session or re-run the pipeline with different parameters)"))
display(Markdown(f"*--> export EXPERIMENT_DIR=data/experiments/{EXPERIMENT_NAME}*"))
display(Markdown(f"*--> export SPLIT={SPLIT}*"))

### 2b. Pseudonymized subset (Do not run for non-pseudonymzed arm)

**Goal:** Build the **pseudonymized** version of an existing non-pseudonymized run, reusing its *exact* `(wikidata_id, summary_id)` selection. It swaps in TMA's pseudonymized summaries (named entities replaced with placeholders like *Entity X* or *Location Y*) and writes them to a new `anon_<source_experiment>` folder, so the two runs never collide.

**Note:** This step does not choose stories or remove duplicates again. It only replaces each summary's text with its pseudonymized version. The dataset stores sentence splits for the original text, not for the pseudonymized text, so we split the pseudonymized text into sentences ourselves. We then re-apply the same two length checks from Section 2 (each summary must fit Llama's `MAX_TOKENS`, and no single sentence may exceed 126 BERT subwords) and the same rule that a story must keep at least `MIN_DEDUPED_EN_SUMMARIES` summaries. Sometimes the new split produces a sentence that is too long for BERT+CRF. When that happens we drop the summary instead of trying to fix it, exactly as Section 2 does. (The dataset names this field `summaries_anonymized`, but the operation is really pseudonymization, so we use that word here.)

**Prerequisite:** Run the Section 0 imports and the Section 2 parameters cell above (they provide the gate constants and `SPLIT`). You do **not** need to run 2a: this cell now mirrors 2a's setup. Point `SOURCE_EXPERIMENT` at the non-pseudonymized run you want to build the pseudonymized version from.

**Then:** Like Section 2, this cell prints two `export` lines (`EXPERIMENT_DIR` and `SPLIT`), but now they point to the pseudonymized folder. It also resets the notebook's `EXPERIMENT_DIR` to that folder, replacing the value Section 2 set. Every later step reads `EXPERIMENT_DIR`, in the notebook and in the terminal, so work on one arm at a time: finish the non-pseudonymized run first, then run this cell and export its lines to point the whole pipeline at the pseudonymized arm. The two runs live in separate folders, so their files never overwrite each other. The only thing that changes is which folder `EXPERIMENT_DIR` points to.

In [ ]:
# Point SOURCE_EXPERIMENT to the experiment name of which you'd like to create a pseudonymized version of
SOURCE_EXPERIMENT = "FILL OUT" # Example: experiment_test_9385_20260625_1059

# Define the path to the raw TellMeAgain zip and the intermediate cache for parsed StoryDataset
TMA_ZIP_PATH = DATA_RAW / "TellMeAgain" / "tell_me_again_v1.zip"
TMA_PARSED_CACHE = DATA_INTERMEDIATE / "_cache_storydataset.pkl"

# Tokenizer: load once and reuse for all summaries (Llama-3-8B-Instruct tokenizer)
if "_tokenizer" not in globals():
    print("Loading Llama-3 tokenizer...")
    _tokenizer = AutoTokenizer.from_pretrained(LLAMA_TOKENIZER_ID)

# Helper function to count Llama tokens in a text
def n_llama_tokens(text: str) -> int:
    return len(_tokenizer.encode(text, add_special_tokens=False))

# BERT tokenizer for the sentence-length filter (matches BERT+CRF training: do_lower_case=True)
if "_bert_tokenizer" not in globals():
    print("Loading BERT tokenizer...")
    _bert_tokenizer = AutoTokenizer.from_pretrained(BERT_TOKENIZER_ID, do_lower_case=True)

# Define a helper function to compute the max BERT subword count across sentences in a summary, which is used for the sentence-length filter.
def max_bert_subwords(sentences):
    """Max subword count across the sentences of a summary (no specials)."""
    return max(
        (len(_bert_tokenizer.encode(s, add_special_tokens=False)) for s in sentences),
        default=0,
    )

# Since TMA StoryDataset is downloaded az a zip and parsing is expensive, include them into in-memory cache for reuse. 
if "_ds" not in globals():
    if TMA_PARSED_CACHE.exists():
        print(f"Loading StoryDataset from disk cache ({TMA_PARSED_CACHE.name})...")
        with open(TMA_PARSED_CACHE, "rb") as f:
            _ds = pickle.load(f)
    else:
        print("Parsing TellMeAgain zip...")
        _ds = StoryDataset(data_path=TMA_ZIP_PATH)
        TMA_PARSED_CACHE.parent.mkdir(parents=True, exist_ok=True)
        with open(TMA_PARSED_CACHE, "wb") as f:
            pickle.dump(_ds, f)
        print(f"  -> cached to {TMA_PARSED_CACHE} "
              f"({TMA_PARSED_CACHE.stat().st_size / 1e6:.1f} MB)")


# Define the file path for the subset
SRC_SUBSET = DATA_EXPERIMENTS / SOURCE_EXPERIMENT / f"tma_subset.{SPLIT}.jsonl"

# Story lookup by numeric id (jsonl stores '1001102'; Story.wikidata_id is 'Q1001102')
by_wid = {s.wikidata_id.lstrip("Q"): s for s in _ds}

# Sentence split for anonymized text: period / ! / ? / ellipsis + whitespace.
# Validated on the corpus: 99.74% of boundaries end in '.', and anonymization strips
# the abbreviations (Jr., Dr., ...) that would otherwise fool a naive period splitter.
def split_anon(text):
    return [x for x in re.split(r'(?<=[.!?…])\s+', text.strip()) if x.strip()]

# Re-apply Section 2's per-summary filters on the pseudonymzed text, collecting per story.
by_story = defaultdict(list)
n_fallback = n_drop_tokens = n_drop_subword = 0
for r in (json.loads(l) for l in open(SRC_SUBSET, encoding="utf-8")):
    story = by_wid.get(str(r["wikidata_id"]).lstrip("Q"))
    anon_text = (story.summaries_anonymized or {}).get(r["summary_id"]) if story else None
    is_anon = bool((anon_text or "").strip())
    if not is_anon:                          # ~2/9385 have no anon text: keep original
        anon_text = r["text"]; n_fallback += 1
    sents = split_anon(anon_text)
    n_tok = n_llama_tokens(anon_text)
    if n_tok > MAX_TOKENS:                                          # Llama context (8192)
        n_drop_tokens += 1; continue
    if max_bert_subwords(sents) > BERT_MAX_SUBWORDS_PER_SENTENCE:   # BERT 126 / sentence
        n_drop_subword += 1; continue
    by_story[r["wikidata_id"]].append({**r,
        "text": anon_text, "sentences": sents,
        "n_sentences": len(sents), "n_tokens": n_tok, "anonymized": is_anon})

# Re-enforce the >=2-summaries-per-story filter (mirror Section 2's MIN_DEDUPED_EN_SUMMARIES).
anon_rows, n_drop_singleton = [], 0
for wid, srows in by_story.items():
    if len(srows) < MIN_DEDUPED_EN_SUMMARIES:
        n_drop_singleton += len(srows); continue
    anon_rows.extend(srows)

# Write to a NEW experiment folder so the two runs never collide.
ANON_NAME       = "anon_" + SOURCE_EXPERIMENT
EXPERIMENT_DIR  = DATA_EXPERIMENTS / ANON_NAME
(EXPERIMENT_DIR / "logs").mkdir(parents=True, exist_ok=True)
TMA_SUBSET_PATH = EXPERIMENT_DIR / f"tma_subset.{SPLIT}.jsonl"
with open(TMA_SUBSET_PATH, "w", encoding="utf-8") as f:
    for row in anon_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

n_stories = len({r["wikidata_id"] for r in anon_rows})

# Print final summary of the subsetting results
print(f"- Wrote {len(anon_rows)} anonymized summaries across {n_stories} stories")
print(f"  ({n_fallback} kept original text; dropped {n_drop_tokens} >8192-tok, "
      f"{n_drop_subword} >126-subword, {n_drop_singleton} singleton-after-drop)")
print(f"- Anon experiment: {ANON_NAME}")
print(f"- Location:        {TMA_SUBSET_PATH}")
print()

# Display the next steps instructions as Markdown for better visibility in the notebook, since these are important to run before proceeding with the Local/Cluster steps.
display(Markdown("**IMPORTANT:** For the Local/Cluster steps below, run this first in the terminal:"))
display(Markdown(f"*--> export EXPERIMENT_DIR=data/experiments/{ANON_NAME}*"))
display(Markdown(f"*--> export SPLIT={SPLIT}*"))

## 3. Event Extraction for TellMeAgain! dataset via BERT+CRF

**Goal:** Run the trained BERT+CRF event-extraction model over every summary in the TMA subset, producing per-event records of the form `{event_id, sent_id, trigger, event_type, start, end}`.

**Setup:**

```bash
# In your terminal, run the two `export` lines printed by Section 2 (they set `EXPERIMENT_DIR` and `SPLIT`) from word-to-word, or fill in your experiment name below and run:
export EXPERIMENT_DIR=data/experiments/<EXPERIMENT_NAME>    # EXPERIMENT_NAME printed by Section 2 above
export SPLIT=test                                           # As printed by Section 2 above

# Inference runs in the BERT+CRF venv (`models/bert_crf/.venv-maven-train`, Python 3.9 + transformers 4.18) because the trained model is incompatible with this notebook's environment. 
# Note that the code prints status updates, ETA etc. once in every 500 summaries. It might be a while to get the inital verbose after running. Check out the log file for crashes, it's written real-time with the code. 
#From the repo root, run: 
models/bert_crf/.venv-maven-train/bin/python models/bert_crf/infer_tma.py \
    --input      $EXPERIMENT_DIR/tma_subset.$SPLIT.jsonl \
    --output     $EXPERIMENT_DIR/tma_subset_events_full.$SPLIT.jsonl \
    --output     $EXPERIMENT_DIR/tma_subset_events_full.$SPLIT.jsonl \
    --output     $EXPERIMENT_DIR/tma_subset_events_full.$SPLIT.jsonl \
    --checkpoint data/intermediate/models/bert_crf \
    2>&1 | tee $EXPERIMENT_DIR/logs/infer_tma_$(date +%Y%m%d_%H%M%S).log

# Optional Test Run: To test the script before the full run, add `--sample-size N` (omit it for the full run):
models/bert_crf/.venv-maven-train/bin/python models/bert_crf/infer_tma.py \
    --input       $EXPERIMENT_DIR/tma_subset.$SPLIT.jsonl \
    --output      $EXPERIMENT_DIR/tma_subset_events_full.$SPLIT.jsonl \
    --checkpoint  data/intermediate/models/bert_crf \
    --sample-size 50
```

**Goal:** Load the BERT+CRF event-extraction output (`tma_subset_events_full.{SPLIT}.jsonl`), produced by the terminal command above, back into the notebook so the next steps can use and inspect it.

In [ ]:
# This file is generated by the inference command in the markdown above, which runs the BERT+CRF event extraction on the subsetted dataset we just created.
TMA_EVENTS_PATH = EXPERIMENT_DIR / f"tma_subset_events_full.{SPLIT}.jsonl"

# Verify that the events file exists before trying to load it, and print instructions if it doesn't.
TMA_EVENTS_PATH = Path(TMA_EVENTS_PATH)
if not TMA_EVENTS_PATH.exists():
    print(f"{TMA_EVENTS_PATH} not found.")
    print("Please run the BERT+CRF inference command from Section 1.2 first, which generates this file by extracting events from the subsetted dataset.")

# Load the events data into a DataFrame and compute the number of events per summary.
events_df       = pd.read_json(TMA_EVENTS_PATH, lines=True)
events_per_summ = events_df["events"].map(len)

# Print summary statistics about the event counts per summary, which is important for understanding the data and for capacity planning (e.g., how many events do we need to handle in the clustering step?).
print(f"Summaries processed: {len(events_df):>9,}")
print(f"Total events:        {events_per_summ.sum():>9,}")
print(f"Empty summaries:     {(events_per_summ == 0).sum():,}")
print()

# Print percentile table to see descriptive statistics about the distribution of event counts per summary
print(f"{'mean':<10}{events_per_summ.mean():>8.1f}")
for p in [50, 75, 90, 95, 99, 99.9, 100]:
    print(f"{'p'+str(p):<10}{events_per_summ.quantile(p/100):>8.1f}")

**Goal:** Spot-check a few random TMA summaries and their extracted events to get a qualitative sense that the extraction worked.

In [ ]:
# Check a few random summaries and their events to get a qualitative sense of the data. If this errors, the inference command in the markdown above hasn't been run yet (or it failed).
from collections import defaultdict
from IPython.display import display, Markdown

# Define a seed for reproducibility; change this to see different random samples of summaries/events.
SAMPLE_SEED = 2

# Take a random sample of 3 summaries from the events DataFrame, and for each summary, group the events by sentence ID and display the sentences with the event triggers highlighted in Markdown format. 
sample = events_df.sample(n=3, random_state=SAMPLE_SEED).to_dict("records")
for row in sample:
    by_sent = defaultdict(list)
    for ev in row["events"]:
        by_sent[ev["sent_id"]].append(ev)
    # Pretty print to capture extraction process clearly
    md = [f"### {row['wikidata_id']} / {row['summary_id']} — {len(row['events'])} events"]
    for sent_id, sent in enumerate(row["sentences"]):
        evs = sorted(by_sent.get(sent_id, []), key=lambda e: e["start"])
        out, cursor = [], 0
        for ev in evs:
            out.append(sent[cursor:ev["start"]])
            out.append(f"**[{ev['trigger']}|{ev['event_type']}]**")
            cursor = ev["end"]
        out.append(sent[cursor:])
        md.append(f"- s{sent_id}: {''.join(out)}")
    display(Markdown("\n".join(md)))


  ### 3.5 Pre-annotation processing

**Goal:** Filter the BERT+CRF output (`tma_subset_events_full.{SPLIT}.jsonl`) down to summaries worth annotating before the LLM step, so no Llama call is spent on faulty inputs. 

  Two filters, applied in order:

  - **Structural-informativeness:** Drop summaries with fewer than `MIN_EVENTS_FOR_RELATIONS = 5` detected events. Too few events cannot support a meaningful relation graph (see Thesis).
  - **Cluster floor:** Re-enforce `MIN_DEDUPED_EN_SUMMARIES = 2` surviving summaries per relevance cluster (`wikidata_id`). A cluster left with a single summary has no gold neighbor is unevaluable for retrieval, so the whole cluster is dropped.

  **Output:** `tma_subset_events_processed.{SPLIT}.jsonl`, which is the input to Section 4.

In [ ]:
# Get the raw events data from the BERT+CRF output
raw = [json.loads(l) for l in TMA_EVENTS_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]

# 1. drop summaries with too few events to support any relation
enough = [r for r in raw if len(r.get("events", [])) >= MIN_EVENTS_FOR_RELATIONS]

# 2. re-enforce >=2 surviving summaries per cluster (drop resulting singletons)
by_wid = collections.defaultdict(list)
for r in enough:
    by_wid[r["wikidata_id"]].append(r)
processed = [r for rs in by_wid.values() if len(rs) >= MIN_DEDUPED_EN_SUMMARIES for r in rs]

# 3. write the processed events file — this is the §4 annotation input
TMA_EVENTS_PROCESSED_PATH = EXPERIMENT_DIR / f"tma_subset_events_processed.{SPLIT}.jsonl"
with TMA_EVENTS_PROCESSED_PATH.open("w", encoding="utf-8") as f:
    for r in processed:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Chcek whether every survivor has >=2 events and belongs to a >=2-summary cluster
assert all(len(r["events"]) >= MIN_EVENTS_FOR_RELATIONS for r in processed)
_c = collections.Counter(r["wikidata_id"] for r in processed)
assert all(v >= MIN_DEDUPED_EN_SUMMARIES for v in _c.values())

# Assign processed events to a DataFrame for the next steps (clustering, annotation, etc.)
events_df_processed = pd.DataFrame(processed)

print(f"Pre-relation-annotation processing: {TMA_EVENTS_PATH.name} -> {TMA_EVENTS_PROCESSED_PATH.name}")
print(f"    -input : {len(raw)} summaries")
print(f"    -Num of summaries dropped since they have <{MIN_EVENTS_FOR_RELATIONS} events: {len(raw)-len(enough):>6}")
print(f"    -Num of summaries dropped since they are singletons: {len(enough)-len(processed):>6}")
print(f"    -Total summaries filtered out   : {len(raw)-len(processed):>6}")
print(f"    -Total summaries/stories left within the processed dataset: {len(processed):>6} summaries / {len(_c):>4} stories")
print(f"    -Output File Location: {TMA_EVENTS_PROCESSED_PATH}")

## 4. Relation Annotation (Llama-3-8B)

**Goal:** Label how the events in each summary relate to each other. For every event pair (source, target) we add a relation, under one of four conditions. Llama prompts can be found in `models/llama/prompts/`.

Three conditions are produced by Llama-3-8B-Instruct, the fourth is built afterwards from the temporal and causal runs, with no Llama call.

| Condition (`--condition`) | How it is built | Output key |
|-|-|-|
| `temporal` | Llama | `temporal_relations` |
| `causal` | Llama | `causal_relations` |
| `temporal_causal_joint` | Llama | `joint_relations` |
| `temporal_causal_independent` | Temporal + causal runs combined (no Llama) | `temporal_relations` + `causal_relations` |

Each relation is `{"source": "e<i>", "target": "e<j>", "relation": "<LABEL>"}`, where the event ids come from Section 3. All relations are stored under `condition_block.relations`.

```bash

#### **Option 1: Run on Local Machine**
```bash
# 1. Export important parameters to be used in the later commands (again), in you loose the terminal session or change experimental parameters/dataset
export EXPERIMENT_DIR=data/experiments/<EXPERIMENT_NAME>     # relative path, same from experiment name in Section 2
export SPLIT=test                                            # or preferred split of TMA, should be already defined in Section 2

# 2. If local compute power allows for LLama run, run the inference for all 3 conditions locally, then run the merger code described below (Steps 1 & 2) locally. 
#   - Optional: Add `--sample-size N` (after `--condition`) to test the scripts for the first N rows; omit for the full run.

#Infers the causal relationships between events
venv/bin/python -m models.llama.infer_relations \
    --input      $EXPERIMENT_DIR/tma_subset_events_processed.${SPLIT}.jsonl \
    --output     $EXPERIMENT_DIR/llama_runs/causal.jsonl \
    --condition  causal

# Infers the temporal relationships between events
venv/bin/python -m models.llama.infer_relations \
    --input      $EXPERIMENT_DIR/tma_subset_events_processed.${SPLIT}.jsonl \
    --output     $EXPERIMENT_DIR/llama_runs/temporal.jsonl \
    --condition  temporal

# Infers the tempero-causal (see CATERs framework in Thesis) relationships between events
venv/bin/python -m models.llama.infer_relations \
    --input      $EXPERIMENT_DIR/tma_subset_events_processed.${SPLIT}.jsonl \
    --output     $EXPERIMENT_DIR/llama_runs/temporal_causal_joint.jsonl \
    --condition  temporal_causal_joint


# Merge temporal and causal relationships (no GPU is needed, can be run in local):

# 3. Compose temporal_causal_independent from temporal + causal
venv/bin/python -m models.llama.compose_tcindep \
    --temporal $EXPERIMENT_DIR/llama_runs/temporal.jsonl \
    --causal   $EXPERIMENT_DIR/llama_runs/causal.jsonl \
    --output   $EXPERIMENT_DIR/llama_runs/temporal_causal_independent.jsonl

# 4. Merge all four per-condition files into the nested experiment.jsonl
venv/bin/python -m src.build_experiment \
    --temporal $EXPERIMENT_DIR/llama_runs/temporal.jsonl \
    --causal   $EXPERIMENT_DIR/llama_runs/causal.jsonl \
    --tcjoint  $EXPERIMENT_DIR/llama_runs/temporal_causal_joint.jsonl \
    --tcindep  $EXPERIMENT_DIR/llama_runs/temporal_causal_independent.jsonl \
    --output   $EXPERIMENT_DIR/experiment.jsonl

```

#### **Option 2: Run on High-Performance Computing Cluster (Snellius)**
All annotation was run on the cluster. **Submit from the repo root** (so `$SLURM_SUBMIT_DIR` resolves) and keep `EXPERIMENT_DIR` **relative**, since an absolute path fails when the compute node tries to write it. 

**Note:** If you already run the pipeline for non-pseudoynmzed arm, skip steps 1 and 6. 

```bash

# 1. clone the git repository to the cluster (run this in cluster login node)
git clone https://github.com/bgrsph/uva-thesis-computational-narrative-analysis.git

# 2. from your local machine (repo root), push the experiment folder up to Snellius so the job can read its input:
rsync -avhP $EXPERIMENT_DIR/ <user>@snellius.surf.nl:<repo-root-on-snellius>/$EXPERIMENT_DIR/

# 3. After logging in the cluster, change your directory to your project root
cd uva-thesis-computational-narrative-analysis

# 4. Export important parameters to be used in the later commands (re-define them if you loose the terminal session or change experimental parameters/dataset)
export EXPERIMENT_DIR=data/experiments/<EXPERIMENT_NAME>     # relative path, same from experiment name in Section 2
export SPLIT=test                                            # or preferred split of TMA, should be already defined in Section 2
export SLURM_PARTITION_GPU=gpu_a100                          # select an available GPU on preference

# 5. Make a directory so that all the llama logs are aggregated under the same roof within cluster
mkdir -p $EXPERIMENT_DIR/llama_runs            

# 6. Setup a new environment within the cluster, use ".venv" as the name.
python3 -m venv .venv                 # this experiment used Python 3.14.5 in local, 3.13.5 in cluster
source .venv/bin/activate             # Windows: venv\Scripts\activate
pip install --upgrade pip
pip install -r requirements.txt      # the repo-root file, NOT models/bert_crf/requirements.txt


# 7. Three Llama jobs: one sbatch per condition, you can execute them rapidly in the terminal, each of them will create sbatch jobs and work seperately. 
#   - Optional: Add `--sample-size N` (after `--condition`) to test the scripts for the first N rows; omit for the full run.)

# Infers the causal relationships between events
sbatch --partition="$SLURM_PARTITION_GPU" --time=72:00:00 -J llama-causal \
models/llama/infer_relations.sbatch \
--input  $EXPERIMENT_DIR/tma_subset_events_processed.$SPLIT.jsonl \
--output $EXPERIMENT_DIR/llama_runs/causal.jsonl --condition causal

# Infers the temporal relationships between events
sbatch --partition="$SLURM_PARTITION_GPU" --time=90:00:00 -J llama-temporal \
models/llama/infer_relations.sbatch \
--input  $EXPERIMENT_DIR/tma_subset_events_processed.$SPLIT.jsonl \
--output $EXPERIMENT_DIR/llama_runs/temporal.jsonl --condition temporal

# Infers the tempero-causal (CATERs framework) relationships between events
sbatch --partition="$SLURM_PARTITION_GPU" --time=72:00:00 -J llama-tcjoint \
models/llama/infer_relations.sbatch \
--input  $EXPERIMENT_DIR/tma_subset_events_processed.$SPLIT.jsonl \
--output $EXPERIMENT_DIR/llama_runs/temporal_causal_joint.jsonl --condition temporal_causal_joint

# 8. Monitor the progress:
squeue -u $USER -i 300                          # Status table refreshes itself once in every 300 seconds (5 minutes)
tail llama-*-*.out                           # Check out all output files to make sure the code is running. Alternatively, check out ".err" files
wc -l $EXPERIMENT_DIR/llama_runs/*.jsonl        # Check out number of rows processed, updates slowly due to large python cache of cluster


# 9. After all batches are done, merge temporal and causal relationships (no GPU is needed, can be run in local and/or login node of cluster):
# Compose temporal_causal_independent from temporal + causal
.venv/bin/python -m models.llama.compose_tcindep \
    --temporal $EXPERIMENT_DIR/llama_runs/temporal.jsonl \
    --causal   $EXPERIMENT_DIR/llama_runs/causal.jsonl \
    --output   $EXPERIMENT_DIR/llama_runs/temporal_causal_independent.jsonl

# 10. Merge all four per-condition files into the nested experiment.jsonl
.venv/bin/python -m src.build_experiment \
    --temporal $EXPERIMENT_DIR/llama_runs/temporal.jsonl \
    --causal   $EXPERIMENT_DIR/llama_runs/causal.jsonl \
    --tcjoint  $EXPERIMENT_DIR/llama_runs/temporal_causal_joint.jsonl \
    --tcindep  $EXPERIMENT_DIR/llama_runs/temporal_causal_independent.jsonl \
    --output   $EXPERIMENT_DIR/experiment.jsonl


# 11. Pull the results back to your machine (run locally, from the repo root):
rsync -avhP <user>@snellius.surf.nl:<repo-root-on-snellius>/$EXPERIMENT_DIR/ $EXPERIMENT_DIR/
```

Per-row schema:
- `condition_block.source` ∈ `{"llama", "composed", "skipped_ctx_overflow"}`
- `condition_block.relations` is `None` on parse failures and on context-window overflow; check `condition_block.parse_error` and
`condition_block.source` to distinguish.
- `condition_block.hit_ctx_cap` is `True` when generation hit `max_new_tokens` mid-output (distinct from `source == "skipped_ctx_overflow"`, which
means input alone overflowed).

**Goal:** Load the per-condition outputs from the Llama runs, which are generated by the inference command in the markdown above that runs the Llama-3-8B-Instruct relation extraction on the BERT+CRF-extracted events. 


In [ ]:
# Map the condition names to the schema keys. 
CONDITION_RELATION_KEYS = {
    "temporal":                       ("temporal_relations",),
    "causal":                         ("causal_relations",),
    "temporal_causal_joint":          ("joint_relations",),
    "temporal_causal_independent":    ("temporal_relations", "causal_relations"),
}

# Store the number of input rows for reference, which includes parse failures and ctx-overflows are written too.
N_INPUT_ROWS = len(events_df_processed)

# Load whatever per-condition outputs exist under data/intermediate/llama_runs/. Each row carries `condition_block`, see the markdown above.
per_condition_frames = {}
rows = []
for cond, keys in CONDITION_RELATION_KEYS.items():
    path = EXPERIMENT_DIR / "llama_runs" / f"{cond}.jsonl"
    if not path.exists():
        rows.append({
            "condition":      cond,
            "rows_out":       "—",
            "source_mix":     "—",
            "parse_fail_pct": "—",
            "rel_total":      "—",
            "rel_per_row":    "(not run)",
        })
        continue

    # If the file exists, load it and compute statistics about the condition blocks, which include the relation counts and parse failure rates.
    df = pd.read_json(path, lines=True)
    per_condition_frames[cond] = df

    # Compute statistics about the condition blocks.
    cb = df["condition_block"]
    sources = cb.map(lambda b: b["source"]).value_counts().to_dict()
    parse_errors = cb.map(lambda b: b.get("parse_error") is not None).sum()

    # Count relations per row, summed across all schema keys for this condition. 
    def _count(rel_dict, keys=keys):
        if not rel_dict:
            return 0
        return sum(len(rel_dict.get(k, []) or []) for k in keys)
    rel_per_row = cb.map(lambda b: _count(b.get("relations")))

    # Append a summary of the statistics for this condition to the rows buffer, which will be printed as a table at the end.
    rows.append({
        "condition":      cond,
        "rows_out":       len(df),
        "source_mix":     ", ".join(f"{k}={v}" for k, v in sorted(sources.items())),
        "parse_fail_pct": f"{100.0 * parse_errors / len(df):.2f}%",
        "rel_total":      int(rel_per_row.sum()),
        "rel_per_row":    f"mean={rel_per_row.mean():.1f}  "
                          f"median={rel_per_row.median():.0f}  "
                          f"max={rel_per_row.max()}",
    })

# Print the summary statistics for each condition in a table, along with the number of input rows for reference. 
print(f"Input rows offered to each condition: {N_INPUT_ROWS:,}")
print()
print(pd.DataFrame(rows).to_string(index=False))

### 4.5 Pre-embedding Processing

**Goal:** Clean the merged `experiment.jsonl` to a *complete-case* corpus before linearization (Section 5) and embedding (Section 6), so every condition is evaluated on the identical set of summaries, such that it's a matched comparison.

Two filters, applied in order:
  - **Complete-case:** Drop any summary where `condition_block.relations is None` in **any** condition. `relations is None` captures both Llama failure modes in one check:
      - **parse errors:** The model's output wasn't valid JSON (e.g. truncated/broken braces);
      - **`skipped_ctx_overflow`:** The input alone exceeded the context window, so no generation ran for relation inference.
- **Cluster floor:** Re-enforce `MIN_DEDUPED_EN_SUMMARIES = 2` surviving summaries per relevance cluster (`wikidata_id`). A cluster reduced to a single summary has no gold neighbor → unevaluable for retrieval, so the whole cluster is dropped.

**Output:** the cleaned `experiment.jsonl` (overwritten in place). The full per-condition outputs — failures included, with raw responses — remain in `$EXPERIMENT_DIR/llama_runs/`, so `experiment.jsonl` is always rebuildable via `src/build_experiment.py`.

In [ ]:
# Get the raw experiment data from the annotation step, which includes the Llama outputs for each condition, along with the relation annotations (or parse errors) in the `condition_block` field.  
EXPERIMENT_PATH  = EXPERIMENT_DIR / "experiment.jsonl"

# List of conditions to check for full annotation (i.e., parseable relations). If a summary is missing relations for any of these conditions, it will be dropped in the complete-case filtering step below.
LLAMA_CONDITIONS = ["temporal", "causal", "temporal_causal_joint"]

# Load the experiment data into memory. Each row corresponds to a summary and contains the Llama outputs and relation annotations for each condition.
rows = [json.loads(l) for l in EXPERIMENT_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
  
# Define a helper function to check if a row is fully annotated, meaning that for every condition in LLAMA_CONDITIONS, the "relations" field in the "condition_block" is not None (i.e., it was successfully parsed and annotated).
def fully_annotated(r):
    return all(r["conditions"][c]["relations"] is not None for c in LLAMA_CONDITIONS)

# 1. complete-case: every Llama condition produced parseable relations
complete = [r for r in rows if fully_annotated(r)]
  
# 2. re-enforce >=2 surviving summaries per relevance cluster (drop new singletons)
by_wid = collections.defaultdict(list)
for r in complete:
    by_wid[r["wikidata_id"]].append(r)
clean = [r for rs in by_wid.values() if len(rs) >= MIN_DEDUPED_EN_SUMMARIES for r in rs]

# 3. overwrite experiment.jsonl (rebuildable from llama_runs via src.build_experiment)
with EXPERIMENT_PATH.open("w", encoding="utf-8") as f:
    for r in clean:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
  
# Check that all surviving rows are fully annotated and that every story has >=MIN_DEDUPED_EN_SUMMARIES summaries, which are the requirements for the final evaluation step. I
# If these assertions fail, it means there's a bug in the filtering logic above that needs to be fixed before proceeding to annotation and evaluation.
assert all(fully_annotated(r) for r in clean)
_c = collections.Counter(r["wikidata_id"] for r in clean)
assert all(v >= MIN_DEDUPED_EN_SUMMARIES for v in _c.values())
  
# Print the summary statistics about the post-annotation filtering steps
print(f"Post-annotation filtering -> {EXPERIMENT_PATH.name}")
print(f"  input                       : {len(rows):>5} summaries")
for c in LLAMA_CONDITIONS:
    nf = sum(1 for r in rows if r['conditions'][c]['relations'] is None)
    print(f"    {c:28s} failed: {nf:>5}")
print(f"  complete-case (all parsed)  : {len(complete):>5}  (dropped {len(rows)-len(complete)})")
print(f"  dropped sub-floor singletons: {len(complete)-len(clean):>5}")
print(f"  -> clean                    : {len(clean):>5} summaries / {len(_c):>4} works")

### 4.6 Drop hallucinated relations (referencing non-existent event ids)

**Goal:** Remove relations that point to an event id the summary does not contain, then save an audit log of everything dropped.

Llama sometimes emits a relation to an id that is not in the summary, usually `eN` to `e(N+1)` when only `e1..eN` exist. `parse_and_validate` only checks the id format (`^e\d+$`), not whether the id exists, so these slip into `experiment.jsonl`. This cell rewrites `experiment.jsonl` in place with the bad triples removed, and writes every dropped triple (with its `(wikidata_id, summary_id)` and some metadata) to a separate `hallucinated_relations.jsonl`, so the audit trail stays out of `experiment.jsonl` and each bad triple can be traced back to its summary for error analysis.


In [ ]:
#Define the experiment path again, and the dropped path to log errors
EXPERIMENT_PATH = EXPERIMENT_DIR / "experiment.jsonl"
DROPPED_PATH    = EXPERIMENT_DIR / "hallucinated_relations.jsonl"

# Define the conditions keys to match with linearization
CONDITION_SUBKEYS = {
    "temporal":                    ("temporal_relations",),
    "causal":                      ("causal_relations",),
    "temporal_causal_joint":       ("joint_relations",),
    "temporal_causal_independent": ("temporal_relations", "causal_relations"),
}

# If event id is not numeric, raise error
def _num(eid):
    try:
        return int(eid[1:])
    except (ValueError, IndexError, TypeError):
        return None

# Download the rows again, we are repeating this to make each cell as independent from each other as possible, and still use the single source of truth, which is experiment.jsonl
rows = [json.loads(l) for l in EXPERIMENT_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
dropped, kept_total, rows_affected = [], 0, 0
for r in rows:
    evset = {e["event_id"] for e in r.get("events", [])}
    max_eid = max((_num(e) for e in evset if _num(e) is not None), default=None)
    had = False
    for cond, subkeys in CONDITION_SUBKEYS.items():
        block = (r.get("conditions") or {}).get(cond)
        rels = block.get("relations") if block else None
        if not rels:
            continue
        for sk in subkeys:
            good = []
            for t in (rels.get(sk) or []):
                missing = [e for e in (t.get("source"), t.get("target")) if e not in evset]
                if missing: # If we find the missig one, record it with meta data
                    had = True
                    dropped.append({
                        "wikidata_id": r["wikidata_id"], "summary_id": r["summary_id"],
                        "condition": cond, "subkey": sk,
                        "source": t.get("source"), "target": t.get("target"), "relation": t.get("relation"),
                        "missing_ids": missing, "n_events": len(evset),
                        "max_event_id": f"e{max_eid}" if max_eid is not None else None,
                        "is_next_in_sequence": bool(max_eid is not None and any(_num(m) == max_eid + 1 for m in missing)),
                    })
                else:   # Else, keep the good ones for later reporting
                    good.append(t)
                    kept_total += 1
            rels[sk] = good
    rows_affected += had

# Write the hallucination-free version to experiment.jsonl
with EXPERIMENT_PATH.open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Log the hallucinations into hallucinated_relations.jsonl
with DROPPED_PATH.open("w", encoding="utf-8") as f:
    for d in dropped:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

# Print out summary of what hapenned
print(f"Hallucinated-relation filter -> {EXPERIMENT_PATH.name}")
print(f"  relations kept               : {kept_total}")
print(f"  relations dropped (dangling) : {len(dropped)}  across {rows_affected} rows")
print(f"  of which eN -> e(N+1) pattern : {sum(1 for d in dropped if d['is_next_in_sequence'])}")
print(f"  audit file                   : {DROPPED_PATH}")

## 5. Linearization


**Goal:** Convert each `(events + relations)` data on `$EXPERIMENT_DIR/experiment.jsonl` into a canonical text format per experimental condition, written back into the same row. 

| Condition | Field on row | Section header(s) emitted |
|---|---|---|
| `events_only` | `linearized_events_only` (top-level) | — (events line only) |
| `event_temporal` | `conditions.temporal.linearized` | `TEMPORAL:` |
| `event_causal` | `conditions.causal.linearized` | `CAUSAL:` |
| `event_temporal_causal_independent` | `conditions.temporal_causal_independent.linearized` | `TEMPORAL:` then `CAUSAL:` |
| `event_temporal_causal_joint` | `conditions.temporal_causal_joint.linearized` | `TEMPEROCAUSAL:` |

Event units are wrapped in parentheses: `(eID|trigger|EVENT_TYPE)`, e.g. `(e1|notices|Know)`. The pipe-delimited content matches the Llama prompt's inlined events, though the prompt uses square brackets (`[eID|trigger|EVENT_TYPE]`) while linearization uses parentheses to keep unit boundaries unambiguous. Relation triples use the form `(source_eID, RELATION, target_eID)`, e.g. `(e3, CAUSE, e4)`. Section headers are always emitted even when the relation list is empty, so condition identity is preserved at the row level.

The StoryEmbed-style instruction prefix (`"Retrieve stories with a similar narrative to the given story."`) is not in the linearized text as it lives in the E5 encoder wrapper at encode time. Hence, one linearization process can be applied to every encoder.

#### **Run linearization:**

```bash
venv/bin/python -m src.linearize --in $EXPERIMENT_DIR/experiment.jsonl --inplace
```

(Here, in place means that experiment.jsonl is both the input file and the output file to keep things organized. If a problem occues mid-run, `experiment.jsonl` can always be rebuilt from `$EXPERIMENT_DIR/llama_runs/*.jsonl` via `src/build_experiment.py`, so a problematic linearization is recoverable.)


In [ ]:
# Load post-linearization experiment.jsonl and report per-condition coverage.
EXPERIMENT_PATH = EXPERIMENT_DIR / "experiment.jsonl"

# Verify that the experiment file exists before trying to load it, since this is the output of the annotation step and may not exist if annotation hasn't been run yet. If it doesn't exist, prompt the user to run the annotation step first.
assert EXPERIMENT_PATH.exists(), f"{EXPERIMENT_PATH} not found — see Section 4 / src/build_experiment.py."

# Load the experiment data into a DataFrame, which includes the Llama outputs for each condition.
experiment_df = pd.read_json(EXPERIMENT_PATH, lines=True)

# Map each condition to a getter function that extracts the linearized string for that condition from the experiment DataFrame, which includes the outputs of the Llama relation extraction for each condition.
LINEARIZATION_GETTERS = {
    "events_only":                       lambda r: r.get("linearized_events_only"),
    "event_temporal":                    lambda r: (r["conditions"]["temporal"]                   or {}).get("linearized"),
    "event_causal":                      lambda r: (r["conditions"]["causal"]                     or {}).get("linearized"),
    "event_temporal_causal_independent": lambda r: (r["conditions"]["temporal_causal_independent"] or {}).get("linearized"),
    "event_temporal_causal_joint":       lambda r: (r["conditions"]["temporal_causal_joint"]      or {}).get("linearized"),
}

# Tag each condition with the header(s) it carries, so we can count empty section bodies (Llama returned no relations of that type for that row).
SECTION_HEADERS = {
    "events_only":                       (),
    "event_temporal":                    ("TEMPORAL:",),
    "event_causal":                      ("CAUSAL:",),
    "event_temporal_causal_independent": ("TEMPORAL:", "CAUSAL:"),
    "event_temporal_causal_joint":       ("TEMPEROCAUSAL:",),
}

# Define a helper function to count how many rows have empty bodies for all section headers of a given condition, which indicates that the Llama model returned no relations for that condition in those rows.
def _empty_body_count(strings, headers):
    # If there are no headers, we can't check for empty bodies, so we return "—" to indicate that this metric is not applicable for this condition.
    if not headers:
        return "—"
    # Otherwise, we iterate over the strings for this condition, and for each string, we check if the body of each header is empty. If all headers have empty bodies, we count that row as having no relations for that condition.
    n = 0
    for s in strings.dropna():
        ok = True
        for i, h in enumerate(headers):
            # Body of header h runs from the line after "<h>\n" until either the next header in `headers` or end-of-string.
            start = s.index(h) + len(h) + 1   # +1 for the newline after "<h>:"
            end   = s.index(headers[i+1]) if i + 1 < len(headers) else len(s)
            body  = s[start:end].strip()
            if body:
                ok = False
                break
        if ok:
            n += 1
    return n

# For each condition, apply the corresponding getter to extract the linearized string, compute how many rows have non-empty strings, and compute statistics about the length of the linearized strings (in characters) for the non-empty rows. 
# Also count how many rows have empty bodies for all section headers of that condition, which indicates no relations were returned for those rows.
stats_rows = []
for cond, getter in LINEARIZATION_GETTERS.items():
    strings = experiment_df.apply(getter, axis=1)
    present = strings.notna()
    lens    = strings[present].str.len()
    stats_rows.append({
        "condition":           cond,
        "rows_present":        f"{present.sum()}/{len(experiment_df)}",
        "len_mean_chars":      f"{lens.mean():.0f}" if not lens.empty else "—",
        "len_p95_chars":       f"{lens.quantile(0.95):.0f}" if not lens.empty else "—",
        "len_max_chars":       f"{lens.max():.0f}" if not lens.empty else "—",
        "all_empty_sections":  _empty_body_count(strings, SECTION_HEADERS[cond]),
    })

# Print the statistics for each condition in a table, which helps us understand the coverage and output length of the Llama relation extraction for each condition.
print(pd.DataFrame(stats_rows).to_string(index=False))

**Goal:** Check random samples to examine how linearization worked

In [ ]:
# Define constants for sampling and previewing the linearized outputs for a few random summaries
N_SAMPLES   = 3     # change to see more rows
PREVIEW     = 600   # change to see more of each row
SAMPLE_SEED = 0     # change to see different rows

# Truncate long linearized strings for better readability in the Markdown display, while still showing the full length of the string for reference. 
# If the string is None, we return a message indicating that the linearization hasn't been run for that row yet.
def _truncate(s, n=PREVIEW):
    if s is None:
        return "(missing — linearization not yet run for this row)"
    if len(s) <= n:
        return s
    return s[:n] + f"\n…(truncated, full length {len(s)} chars)"

# Take a random sample of rows from the experiment DataFrame, and for each row, display the Wikidata ID, summary ID, language, number of events, and the linearized outputs for each condition in a Markdown format.
sample = experiment_df.sample(n=min(N_SAMPLES, len(experiment_df)), random_state=SAMPLE_SEED)

# For each sampled row, we create a Markdown string that includes the Wikidata ID, summary ID, language, and number of events. Then for each condition, we append the linearized output for that condition (truncated for readability) to the Markdown string. 
# Finally, we display the Markdown for each sampled row.
for _, row in sample.iterrows():
    md = [f"### {row['wikidata_id']} / {row['summary_id']} ({row['lang']}) — {len(row['events'])} events"]
    md.append("**events_only**\n```\n" + _truncate(LINEARIZATION_GETTERS["events_only"](row)) + "\n```")
    for cond in ("event_temporal", "event_causal", "event_temporal_causal_independent", "event_temporal_causal_joint"):
        md.append(f"**{cond}**\n```\n" + _truncate(LINEARIZATION_GETTERS[cond](row)) + "\n```")
    display(Markdown("\n\n".join(md)))


### 5.5 Pre-embedding: drop linearized-overflow summaries, then re-enforce the cluster floor

**Goal:** Keep one identical evaluation set across every encoder and condition by dropping any summary whose linearized text would overflow the embedder's 4096-token window, then re-checking the minimum-summaries-per-work floor.

**Why:** E5-Mistral and StoryEmbed cap at `max_seq_length = 4096` and truncate on the right. For long structured conditions this silently cuts the relation tail (`TEMPORAL:` / `CAUSAL:`) and collapses them back toward `events_only`, which would bias the results against the structure hypothesis on exactly the longest, most event-dense summaries. Qwen3 has a 32k window and never truncates, but it is held to the same corpus so all conditions stay comparable (controlled comparison, Thesis Section 3.2.4).

**Note:** This overwrites `experiment.jsonl` in place, and the drop statistics are appended to `experiment.yaml`.

In [ ]:
# Import the registry from our source code
from models.embed.encoders import ENCODER_REGISTRY
from models.embed.encode_embeddings import CONDITION_KEYS, _gather_inputs

# Define the constants
EXPERIMENT_PATH = EXPERIMENT_DIR / "experiment.jsonl"
EXPERIMENT_YAML = EXPERIMENT_DIR / "experiment.yaml"
CAP_E5 = 4096   # E5-Mistral / StoryEmbed effective max_seq_length (verified on the cluster)

# Get the list of condition keys that are linearized and fed to E5.
# raw_text is excluded: it is not a linearized representation (and since none of them is abover 4096 token limit).
LINEARIZED_KEYS = tuple(c for c in CONDITION_KEYS if c != "raw_text")

# Reuse the registry's model_id + task so the token count matches what E5 actually encodes.
E5_MODEL_ID, E5_TASK = ENCODER_REGISTRY["e5_mistral"]
if "_e5_tokenizer" not in globals():
    print(f"Loading E5 tokenizer ({E5_MODEL_ID})...")
    _e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_ID)

# Define function to return True if any of the linearized conditions for a given row exceeds the token limit of E5
def _row_overflows_e5(row) -> bool:
    """True if any linearized condition's tokenized input (prefix + specials) exceeds CAP_E5."""
    texts = [f"Instruct: {E5_TASK}\nQuery: {t}" for t in _gather_inputs(row, LINEARIZED_KEYS)]
    ids = _e5_tokenizer(texts, add_special_tokens=True)["input_ids"]
    return any(len(x) > CAP_E5 for x in ids)

# Load the experiment data into memory, which includes the Llama outputs and relation annotations for each condition
rows = [json.loads(l) for l in EXPERIMENT_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
n_in     = len(rows)
works_in = len({r["wikidata_id"] for r in rows})

# 1. Drop summaries whose linearized representation overflows E5's window.
kept = [r for r in rows if not _row_overflows_e5(r)]
n_dropped_overflow = n_in - len(kept)

# 2. Re-enforce >=MIN_DEDUPED_EN_SUMMARIES per work (single pass; clusters disjoint by wikidata_id).
by_wid = collections.defaultdict(list)
for r in kept:
    by_wid[r["wikidata_id"]].append(r)
clean = [r for rs in by_wid.values() if len(rs) >= MIN_DEDUPED_EN_SUMMARIES for r in rs]
n_dropped_singletons = len(kept) - len(clean)
works_out = len({r["wikidata_id"] for r in clean})

# 3. Overwrite experiment.jsonl atomically (same pattern as §4.5 / §4.6).
tmp = EXPERIMENT_PATH.with_suffix(EXPERIMENT_PATH.suffix + ".tmp")
with tmp.open("w", encoding="utf-8") as f:
    for r in clean:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
os.replace(tmp, EXPERIMENT_PATH)

# 4. Append drop statistics to experiment.yaml (merge — don't clobber other sections).
stats = {
    "pre_embedding_e5_truncation_filter": {
        "cap_tokens": CAP_E5,
        "bound_by_encoders": ["e5_mistral", "story_emb"],
        "linearized_conditions_checked": list(LINEARIZED_KEYS),
        "min_summaries_per_work": MIN_DEDUPED_EN_SUMMARIES,
        "summaries_in": n_in,
        "summaries_dropped_overflow": n_dropped_overflow,
        "summaries_dropped_new_singletons": n_dropped_singletons,
        "summaries_out": len(clean),
        "works_in": works_in,
        "works_out": works_out,
    }
}
doc = yaml.safe_load(EXPERIMENT_YAML.read_text(encoding="utf-8")) or {} if EXPERIMENT_YAML.exists() else {}
doc.update(stats)
EXPERIMENT_YAML.write_text(yaml.safe_dump(doc, sort_keys=False, allow_unicode=True), encoding="utf-8")

# 5. Reporr the restuls
print(f"§5.5 E5-truncation filter (cap={CAP_E5}, linearized conditions) -> {EXPERIMENT_PATH.name}")
print(f"  input                        : {n_in:>5} summaries / {works_in:>4} works")
print(f"  dropped (>{CAP_E5} E5 tokens)   : {n_dropped_overflow:>5}")
print(f"  dropped (new singletons)     : {n_dropped_singletons:>5}")
print(f"  -> clean                     : {len(clean):>5} summaries / {works_out:>4} works")
print(f"  stats written to             : {EXPERIMENT_YAML.name}")


 ## 6. Embedding

**Goal:** Encode each row's six condition inputs with each registered encoder, then merge the per-encoder vectors back into `experiment.jsonl` (read by Section 8 - retrieval). All vectors are L2-normalized, so cosine = dot product downstream.

  **Inputs per condition:**

  | Condition key | Input string |
  |---|---|
  | `raw_text` | `row["text"]` |
  | `events_only` | `row["linearized_events_only"]` |
  | `temporal` | `row["conditions"]["temporal"]["linearized"]` |
  | `causal` | `row["conditions"]["causal"]["linearized"]` |
  | `temporal_causal_independent` | `row["conditions"]["temporal_causal_independent"]["linearized"]` |
  | `temporal_causal_joint` | `row["conditions"]["temporal_causal_joint"]["linearized"]` |

  `e5_mistral` and `qwen3_emb_0p6b` embed all six conditions, `story_emb` embeds only `raw_text`. Each encoder writes to `$EXPERIMENT_DIR/embeddings/<encoder>.jsonl`.  Then, we merge them via `src/build_embeddings.py` into `experiment.jsonl::embeddings.<encoder>`. Similar to others, this has been engineered this way to protect the encoding results, as cluster operations can be costly. 

#### **Option 1: Run on Local Machine**
If computation power allows for the embedding models, run via local machine is possible. 

```bash

# EXPERIMENT_DIR as printed in Section 2 (re-define if the terminal session is lost)
export EXPERIMENT_DIR=data/experiments/<EXPERIMENT_NAME>

# 1. Create embeddings via e5_mistral model(reads experiment.jsonl, writes embeddings/<encoder>.jsonl)
venv/bin/python -m models.embed.encode_embeddings --encoder e5_mistral \
    --input $EXPERIMENT_DIR/experiment.jsonl --output $EXPERIMENT_DIR/embeddings/e5_mistral.jsonl

# 2. Create embeddings via Qwen3-0.6B model(reads experiment.jsonl, writes embeddings/<encoder>.jsonl)
venv/bin/python -m models.embed.encode_embeddings --encoder qwen3_emb_0p6b \
    --input $EXPERIMENT_DIR/experiment.jsonl --output $EXPERIMENT_DIR/embeddings/qwen3_emb_0p6b.jsonl

# 3. Create embeddings via StoryEmbed model(reads experiment.jsonl, writes embeddings/<encoder>.jsonl)
venv/bin/python -m models.embed.encode_embeddings --encoder story_emb \
    --input $EXPERIMENT_DIR/experiment.jsonl --output $EXPERIMENT_DIR/embeddings/story_emb.jsonl

# 4. Merge all encoder outputs into experiment.jsonl (in place)
venv/bin/python -m src.build_embeddings --in $EXPERIMENT_DIR/experiment.jsonl \
    --encoders $EXPERIMENT_DIR/embeddings/e5_mistral.jsonl \
                $EXPERIMENT_DIR/embeddings/qwen3_emb_0p6b.jsonl \
                $EXPERIMENT_DIR/embeddings/story_emb.jsonl \
    --inplace

```

#### **Option 2: Run on High-Performance Computing Cluster (Snellius)**

This experiment also ran within the cluster, due to computational limitations on local machines. 

```bash


# 1. First, from your local machine (repo root), push the updated experiment.jsonl up to Snellius (since local Sections 4.5 to 5.5 changed it):
rsync -avhP $EXPERIMENT_DIR/experiment.jsonl <user>@snellius.surf.nl:<repo-root-on-snellius>/$EXPERIMENT_DIR/experiment.jsonl

# 2. After logging in to cluster, change your directory to the project root
cd <repo-root-on-snellius>

# 3. Export parameters (re-define if the terminal session is lost)
export EXPERIMENT_DIR=data/experiments/<EXPERIMENT_NAME>     # relative path, same as Section 2
export SLURM_PARTITION_GPU=gpu_a100                          # available and preferable GPU partition


# 4. Create embeddings via e5_mistral model(reads experiment.jsonl, writes embeddings/<encoder>.jsonl)
sbatch --partition="$SLURM_PARTITION_GPU" --time=8:00:00 -J embed-e5_mistral \
    models/embed/encode_embeddings.sbatch \
    --encoder e5_mistral --input $EXPERIMENT_DIR/experiment.jsonl \
    --output $EXPERIMENT_DIR/embeddings/e5_mistral.jsonl

# 4. Create embeddings via Qwen3-0.6B model(reads experiment.jsonl, writes embeddings/<encoder>.jsonl)
sbatch --partition="$SLURM_PARTITION_GPU" --time=6:00:00 -J embed-qwen3_emb_0p6b \
    models/embed/encode_embeddings.sbatch \
    --encoder qwen3_emb_0p6b --input $EXPERIMENT_DIR/experiment.jsonl \
    --output $EXPERIMENT_DIR/embeddings/qwen3_emb_0p6b.jsonl

# 4. Create embeddings via StoryEmbed model(reads experiment.jsonl, writes embeddings/<encoder>.jsonl)
sbatch --partition="$SLURM_PARTITION_GPU" --time=6:00:00 -J embed-story_emb \
    models/embed/encode_embeddings.sbatch \
    --encoder story_emb --input $EXPERIMENT_DIR/experiment.jsonl \
    --output $EXPERIMENT_DIR/embeddings/story_emb.jsonl

# 5. Monitor the progress:
squeue -u $USER -i 300                       # status table refreshes every 300 seconds (5 minutes)
tail embed-*-*.out                        # check outputs are running, or ".err" for warnings/tracebacks
wc -l $EXPERIMENT_DIR/embeddings/*.jsonl     # gives the number of rows processed (updates infrequently due to python cache size of cluster)

# 6. Merge all encoder outputs into experiment.jsonl (no GPU; run on the login node)
.venv/bin/python -m src.build_embeddings --in $EXPERIMENT_DIR/experiment.jsonl \
    --encoders $EXPERIMENT_DIR/embeddings/e5_mistral.jsonl \
                $EXPERIMENT_DIR/embeddings/qwen3_emb_0p6b.jsonl \
                $EXPERIMENT_DIR/embeddings/story_emb.jsonl \
    --inplace

# 7. Pull the embedded experiment.jsonl back to your machine (run locally, from the repo root)
rsync -avhP <user>@snellius.surf.nl:<repo-root-on-snellius>/$EXPERIMENT_DIR/ $EXPERIMENT_DIR/
```

**Note:** In case of a problem, re-run any single encoder independently and re-run the merge since `build_embeddings` is idempotent

**Goal:** Inspect the aggregate statistics and make sure encoding worked: presence per encoder, dim, L2-norm distribution, empty-input counts.

In [ ]:
# Load the experiment data again (in case it was overwritten by the complete-case filtering step above)
rows = [json.loads(l) for l in (EXPERIMENT_DIR / "experiment.jsonl").read_text().splitlines() if l]

# Get the set of all encoders that produced embeddings in the experiment data, which is used to compute statistics about the embeddings for each condition.
encoders = sorted({k for r in rows for k in r.get("embeddings", {}).keys()})

# Define a helper function to check if the input for a given condition is empty for a row, which is used to count how many rows have empty inputs for each condition. 
# The definition of "empty" depends on the condition: for "raw_text", it's whether the text is empty; for "events_only", it's whether the linearized events section is empty; for the other conditions, it's whether all section bodies are empty.
def _empty_input(row, cond):
    if cond == "raw_text":
        return not row["text"].strip()
    if cond == "events_only":
        return row["linearized_events_only"].strip() == "EVENTS:"
    return row["conditions"][cond]["linearized"].count("\n") <= 2

# For each encoder, we compute how many rows have embeddings for that encoder, the dimensions of the embeddings, the distribution of L2 norms of the embedding vectors for each condition, and the count of empty inputs for each condition.
for enc in encoders:
    # get the present rows for this encoder
    present = [r for r in rows if enc in r.get("embeddings", {})]

    # get the set of embedding dimensions declared by the present rows for this encoder
    dims = {r["embeddings"][enc]["dim"] for r in present}

    # Each encoder declares its own condition coverage via the JSONL it wrote, iterate over keys actually present rather than the canonical six-tuple.
    enc_conditions = tuple(present[0]["embeddings"][enc]["vectors"].keys()) if present else ()

    # Print the statistics for this encoder, including the number of rows with embeddings, the embedding dimensions, and for each condition, the distribution of L2 norms and the count of empty inputs.
    print(f"\n[{enc}] rows_present = {len(present)}/{len(rows)}, dim = {dims}")
    for cond in enc_conditions:
        norms = np.array([
            float(np.linalg.norm(r["embeddings"][enc]["vectors"][cond]))
            for r in present
        ])
        empty_inputs = sum(1 for r in present if _empty_input(r, cond))
        if len(norms) == 0:
            print(f"  {cond:32s} (no rows)  empty_inputs={empty_inputs}")
            continue
        print(f"  {cond:32s} norm min={norms.min():.6f} max={norms.max():.6f} "
              f"mean={norms.mean():.6f}  empty_inputs={empty_inputs}")

# If no encoders were found in the experiment data, print a message prompting the user to run the BERT+CRF inference step to generate the embeddings before proceeding with the clustering and evaluation steps.
if not encoders:
    print("No encoders populated yet — submit the sbatch jobs above and rerun the merger first.")


**Goal:** Investigate random samples, that has the rendered input strings and first 8 dimensions of each vector per encoder

In [ ]:
# Define the seed, change it to see different samples
random.seed(0)

# Get the rows from the experiment data again (in case it was overwritten by the complete-case filtering step above)
#rows = [json.loads(l) for l in (EXPERIMENT_DIR / "experiment.jsonl").read_text().splitlines() if l]

# Take a random sample of rows to inspect the inputs and embeddings for each condition
sample = random.sample(rows, k=min(2, len(rows)))

# Define a mapping from condition names to functions that extract the corresponding input string for that condition from a row, which is used to display the inputs for each condition in the sampled rows.
CONDITION_INPUTS = {
    "raw_text":                    lambda r: r["text"],
    "events_only":                 lambda r: r["linearized_events_only"],
    "temporal":                    lambda r: r["conditions"]["temporal"]["linearized"],
    "causal":                      lambda r: r["conditions"]["causal"]["linearized"],
    "temporal_causal_independent": lambda r: r["conditions"]["temporal_causal_independent"]["linearized"],
    "temporal_causal_joint":       lambda r: r["conditions"]["temporal_causal_joint"]["linearized"],
}

# For each sampled row, print the Wikidata ID and summary ID, then for each condition that has embeddings in that row, print the first 200 characters of the input for that condition and the first 8 dimensions of the embedding vector
for r in sample:
    print(f"\n=== {r['wikidata_id']}/{r['summary_id']} ===")
    # Union of all condition keys actually embedded across encoders for this row.
    all_conds = sorted(
        {c for enc_block in r.get("embeddings", {}).values() for c in enc_block["vectors"].keys()},
        key=lambda c: list(CONDITION_INPUTS).index(c),
    )
    # For each condition that has embeddings, print the input and the first 8 dimensions of the embedding vector for each encoder that embedded that condition
    for cond in all_conds:
        print(f"\n  -- {cond} --")
        print(f"  input[:200]: {CONDITION_INPUTS[cond](r)[:2000].replace(chr(10), ' / ')!r}")
        for enc, block in r.get("embeddings", {}).items():
            if cond not in block["vectors"]:
                continue  # encoder did not embed this condition (e.g. story_emb on linearized inputs)
            vec = block["vectors"][cond]
            print(f"  {enc} first 8 dims: {[round(x, 4) for x in vec[:8]]}")
